# Customer Segmentation Analysis — RFM + K-Means Clustering
**Objective:** Segment an e-commerce company's customer base into distinct groups based on
purchasing behaviour (Recency, Frequency, Monetary value) using K-Means clustering, to enable
targeted marketing strategies.

**Dataset used:** *(fill in — recommended: "Online Retail" dataset from UCI / Kaggle)*

**Tech stack:** Python, pandas, scikit-learn (KMeans), matplotlib, seaborn


## Step 0 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)


## Step 1 — Load Dataset & Initial Inspection
The classic "Online Retail" dataset (UCI/Kaggle) has these columns:
`InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country`.
If your dataset differs, adjust the column names in the cells below (`customer_col`,
`invoice_col`, `date_col`, `quantity_col`, `price_col`).


In [ ]:
# --- Load the data ---
df = pd.read_csv('online_retail.csv', encoding='latin1')  # switch to pd.read_excel(...) if .xlsx
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# --- Null value check ---
null_summary = df.isnull().sum().sort_values(ascending=False)
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
pd.DataFrame({'missing_count': null_summary, 'missing_pct': null_pct.round(2)})[lambda x: x['missing_count'] > 0]


**Observation (fill in):** Note which columns have missing values. The classic Online Retail
dataset typically has a large chunk of missing `CustomerID` — these rows are unusable for
per-customer segmentation and must be dropped.


In [ ]:
customer_col = 'CustomerID'   # <-- change if different
invoice_col  = 'InvoiceNo'    # <-- change if different
date_col     = 'InvoiceDate'  # <-- change if different
quantity_col = 'Quantity'     # <-- change if different
price_col    = 'UnitPrice'    # <-- change if different

# --- Handle missing values and inconsistent data ---
df_clean = df.dropna(subset=[customer_col]).copy()

# Convert date
df_clean[date_col] = pd.to_datetime(df_clean[date_col], errors='coerce')

# Remove cancelled orders / returns (Online Retail marks these with InvoiceNo starting with 'C')
# and remove non-positive quantity or price rows (data entry errors / returns)
df_clean = df_clean[~df_clean[invoice_col].astype(str).str.startswith('C')]
df_clean = df_clean[(df_clean[quantity_col] > 0) & (df_clean[price_col] > 0)]

# Drop exact duplicates
df_clean = df_clean.drop_duplicates()

# Total line revenue per transaction row
df_clean['LineTotal'] = df_clean[quantity_col] * df_clean[price_col]

print("Rows before cleaning:", df.shape[0])
print("Rows after cleaning:", df_clean.shape[0])
print("Unique customers:", df_clean[customer_col].nunique())


## Step 2 — Descriptive Statistics
Average purchase value, purchase frequency, and a first look at customer lifetime value (CLV).


In [ ]:
# Average purchase (line item) value
avg_purchase_value = df_clean['LineTotal'].mean()
print(f"Average purchase (line) value: {avg_purchase_value:.2f}")

# Purchase frequency per customer = number of distinct invoices
purchase_freq = df_clean.groupby(customer_col)[invoice_col].nunique()
print("\nPurchase frequency per customer (distinct orders):")
print(purchase_freq.describe())

# Customer Lifetime Value (simple definition: total revenue per customer to date)
clv = df_clean.groupby(customer_col)['LineTotal'].sum()
print("\nCustomer Lifetime Value (total revenue to date):")
print(clv.describe())


**Observation (fill in):** Is CLV skewed (a few high-value customers vs. many low-value ones)?
How often does a typical customer purchase?


## Step 3 — Feature Selection: RFM Analysis
We build the three classic behavioural features used for customer segmentation:
- **Recency** — days since the customer's last purchase (lower = more recently active)
- **Frequency** — number of distinct purchases (orders)
- **Monetary** — total amount spent


In [ ]:
# Reference date = one day after the most recent transaction in the dataset
snapshot_date = df_clean[date_col].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby(customer_col).agg(
    Recency=(date_col, lambda x: (snapshot_date - x.max()).days),
    Frequency=(invoice_col, 'nunique'),
    Monetary=('LineTotal', 'sum')
).reset_index()

print("RFM table shape:", rfm.shape)
rfm.head()


In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe()


**Observation (fill in):** Which of the 3 features shows the widest spread? Any obvious
outliers (e.g. one customer with extremely high Monetary value)?


## Step 4 — Data Normalisation / Standardisation
K-Means is distance-based, so features on very different scales (Recency in days vs. Monetary
in currency) must be standardised first — otherwise Monetary would dominate the clustering.


In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
X = rfm[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features)

X_scaled.describe()


## Step 5 — K-Means Clustering: Elbow Method to Find Optimal K


In [ ]:
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(K_range, inertia, marker='o')
plt.title('Elbow Method — Optimal Number of Clusters (K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.xticks(list(K_range))
plt.tight_layout()
plt.show()


**Observation (fill in):** Look at the chart above — where does the curve start to flatten out
("the elbow")? That's your optimal K. Commonly for RFM segmentation this is **K=4** or **K=5**;
update `optimal_k` below to match what you actually see in your chart.


In [ ]:
optimal_k = 4   # <-- set this based on the elbow chart above

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(X_scaled)

rfm.head()


## Step 6 — Visualise Clusters (Scatter Plots)
Two different feature-pair views of the same clusters.


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', palette='tab10', s=60, alpha=0.7)
plt.title('Customer Clusters — Recency vs. Monetary')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster', palette='tab10', s=60, alpha=0.7)
plt.title('Customer Clusters — Frequency vs. Monetary')
plt.tight_layout()
plt.show()


**Observation (fill in):** Do the clusters look visually well-separated? Which cluster sits in
the "high Monetary, low Recency, high Frequency" corner (your best customers)?


## Step 7 — Profile Each Cluster
Mean feature values per cluster, to describe each customer type in plain business terms.


In [ ]:
cluster_profile = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
cluster_profile['Customer Count'] = rfm['Cluster'].value_counts().sort_index()
cluster_profile


**Observation (fill in):** For each cluster number, write a one-line description, e.g.:
- Cluster 0: Low recency, high frequency, high monetary → **Loyal / VIP customers**
- Cluster 1: High recency, low frequency, low monetary → **At-risk / churned customers**
- Cluster 2: Low recency, low frequency, low monetary → **New customers**
- Cluster 3: Moderate recency/frequency/monetary → **Average / occasional shoppers**

(Match these to your *actual* `cluster_profile` numbers above — the labels above are illustrative.)


## Step 8 — Number of Customers per Cluster

In [ ]:
plt.figure(figsize=(8, 5))
rfm['Cluster'].value_counts().sort_index().plot(kind='bar', color='slateblue')
plt.title('Number of Customers per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Observation (fill in):** Is customer distribution across clusters balanced, or is one segment
(e.g. "average shoppers") much larger than the others (e.g. "VIPs")?


## Step 9 — Insights: Marketing Recommendations per Segment

For each cluster identified above, write a specific, actionable marketing recommendation.

**Example structure (replace with your actual cluster numbers/labels):**

- **Cluster 0 — Loyal / VIP customers:** Low recency, high frequency, high monetary.
  *Recommendation:* Reward with a loyalty program, early access to new products, and
  personalised thank-you offers to maintain retention.

- **Cluster 1 — At-risk / churned customers:** High recency, low frequency, low monetary.
  *Recommendation:* Launch a win-back email campaign with a limited-time discount to
  re-engage before they're lost entirely.

- **Cluster 2 — New customers:** Recent first purchase, low frequency so far.
  *Recommendation:* Onboarding email series and a second-purchase incentive to build the
  habit of repeat buying.

- **Cluster 3 — Average / occasional shoppers:** Moderate across all three RFM dimensions.
  *Recommendation:* Targeted cross-sell/upsell campaigns based on past purchase category to
  increase frequency and order value.

*(Rewrite the above using your own cluster numbers and actual mean RFM values from Step 7.)*


## Conclusion

**Summary of key findings:**
1. *(fill in)*
2. *(fill in)*
3. *(fill in)*

**Business impact:** Segmenting customers via RFM + K-Means allows the marketing team to move
away from one-size-fits-all campaigns and instead tailor messaging, offers, and channel spend
to each behavioural segment — improving retention of high-value customers and win-back rates
for at-risk ones.
